In [1]:
pip install pyspark pandas

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: pyspark in c:\users\hp\spark-env\lib\site-packages (4.1.1)
  Using cached pandas-2.3.3-cp310-cp310-win_amd64.whl (11.3 MB)




[notice] A new release of pip available: 22.2.2 -> 26.0.1
[notice] To update, run: C:\Users\hp\spark-env\Scripts\python.exe -m pip install --upgrade pip


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Ecommerce_Analytics") \
    .getOrCreate()

In [3]:
base_path = "C:/Users/hp/OneDrive/Desktop/pyspark_project/Data/"

In [4]:
def load_csv(file):
    return spark.read.csv(
        base_path + file,
        header=True,
        inferSchema=True
    )

orders = load_csv("olist_orders_dataset.csv")
customers = load_csv("olist_customers_dataset.csv")
order_items = load_csv("olist_order_items_dataset.csv")
payments = load_csv("olist_order_payments_dataset.csv")
products = load_csv("olist_products_dataset.csv")
sellers = load_csv("olist_sellers_dataset.csv")
reviews = load_csv("olist_order_reviews_dataset.csv")

In [5]:
orders.show(5)
orders.printSchema()
orders.count()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

99441

In [6]:
orders = orders.dropDuplicates()
customers = customers.dropDuplicates()
order_items = order_items.dropDuplicates()

In [7]:
from pyspark.sql.functions import to_timestamp, col

orders = orders.withColumn(
    "order_purchase_timestamp",
    to_timestamp(col("order_purchase_timestamp"))
)

orders = orders.withColumn(
    "order_delivered_customer_date",
    to_timestamp(col("order_delivered_customer_date"))
)

orders = orders.withColumn(
    "order_estimated_delivery_date",
    to_timestamp(col("order_estimated_delivery_date"))
)

In [8]:
orders = orders.fillna({"order_status": "unknown"})
reviews = reviews.fillna({"review_score": 0})

In [9]:
df = orders \
    .join(customers, "customer_id", "left") \
    .join(order_items, "order_id", "left") \
    .join(payments, "order_id", "left") \
    .join(products, "product_id", "left") \
    .join(sellers, "seller_id", "left") \
    .join(reviews, "order_id", "left")

In [10]:
from pyspark.sql.functions import datediff

df = df.withColumn(
    "delivery_delay_days",
    datediff(
        col("order_delivered_customer_date"),
        col("order_estimated_delivery_date")
    )
)

In [11]:
from pyspark.sql.functions import when

df = df.withColumn(
    "is_delivered",
    when(col("order_status") == "delivered", 1).otherwise(0)
)

In [12]:
from pyspark.sql.functions import year, month

df = df.withColumn("year", year("order_purchase_timestamp"))
df = df.withColumn("month", month("order_purchase_timestamp"))

In [13]:
df.selectExpr("sum(payment_value) as total_revenue").show()

+-------------------+
|      total_revenue|
+-------------------+
|2.057966400999997E7|
+-------------------+



In [14]:
df.select("order_id").distinct().count()

99441

In [15]:
df.selectExpr("avg(payment_value)").show()

+------------------+
|avg(payment_value)|
+------------------+
|172.73513521906978|
+------------------+



In [17]:
import os

output_path = base_path + "output/"

os.makedirs(output_path, exist_ok=True)

df.limit(100000).toPandas().to_csv(
    output_path + "fact_orders.csv",
    index=False
)